# 27 — MLS 2.5D Ordinal Threshold Model, EfficientNet-B4

## Goal

A higher-capacity version of Notebook 24.

Same target formulation:
- MLS >= 1 mm
- MLS >= 3 mm
- MLS >= 5 mm
- relevant-slice probability

Changes:
- EfficientNet-B4 instead of B2
- 512x512 input instead of 384x384
- more samples per epoch
- smaller batch size

The locked TEST split is not used.


## 1. Environment

In [ ]:
!pip install -q pydicom pylibjpeg pylibjpeg-libjpeg

## 2. Imports

In [ ]:
import json, random, warnings
from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F
from pydicom.pixels import apply_modality_lut
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", message="Invalid value for VR UI.*")

## 3. Configuration

In [ ]:
SEED=20260918
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TEST,N_DEV,N_SPLIT_TRIALS=68,54,1500
IMAGE_SIZE=512
BRAIN_CENTER,BRAIN_WIDTH=40.0,80.0
BATCH_SIZE,NUM_WORKERS=6,2
EPOCHS,PATIENCE=20,6
SAMPLES_PER_EPOCH=7000
LR,WEIGHT_DECAY=2e-4,1e-4
CONTEXT_OFFSETS=(-1,0,1)
HU_CACHE_SIZE=128

DATASET_ROOT_CANDIDATES=[Path("/kaggle/input/datasets/mehdipaykanheyrati/iaaa-contest-bct"),Path("/kaggle/input/iaaa-contest-bct")]
OUTPUT_ROOT=Path("/kaggle/working/mls_2p5d_ordinal_b4")
MODELS_DIR=OUTPUT_ROOT/"models"; METRICS_DIR=OUTPUT_ROOT/"metrics"; CACHE_DIR=OUTPUT_ROOT/"cache"
for p in [MODELS_DIR,METRICS_DIR,CACHE_DIR]: p.mkdir(parents=True,exist_ok=True)
print("Device:",DEVICE)

## 4. Locate data and reconstruct common split

In [ ]:
def first_existing(paths): return next((p for p in paths if p.exists()),None)
def norm_id(v):
    try: return str(int(float(v)))
    except Exception: return str(v).strip()

DATASET_ROOT=first_existing(DATASET_ROOT_CANDIDATES)
if DATASET_ROOT is None: raise FileNotFoundError("Competition dataset not found.")
DATA_ROOT=first_existing([DATASET_ROOT/"iaaa-contest-bct"/"Data",DATASET_ROOT/"Data"])
TRAINING_DIR,ANNOTATIONS_DIR=DATA_ROOT/"training",DATA_ROOT/"annotations"
TARGETS_PATH=first_existing([DATASET_ROOT/"series_targets_df.csv",DATASET_ROOT/"iaaa-contest-bct"/"series_targets_df.csv",DATA_ROOT/"series_targets_df.csv",DATA_ROOT.parent/"series_targets_df.csv"])
targets=pd.read_csv(TARGETS_PATH).drop(columns=["Unnamed: 0"],errors="ignore"); targets["series_id"]=targets["series_id"].map(norm_id)
for c in ["V_EDH","V_SDH","V_IPH","V_SAH","V_IVH","fracture_prob","MLS_mm"]: targets[c]=pd.to_numeric(targets[c],errors="raise")
targets["triage_class"]=pd.to_numeric(targets["triage_class"],errors="raise").astype(int)

ICH=["V_EDH","V_SDH","V_IPH","V_SAH","V_IVH"]
def features(df):
    x=df.copy(); total=x[ICH].sum(axis=1); x["feature_any_ich"]=(total>=.1).astype(int); x["feature_fracture"]=(x.fracture_prob>=.5).astype(int)
    for c in ICH: x[f"feature_{c}"]=(x[c]>=.1).astype(int)
    x["mls_bin"]=pd.cut(x.MLS_mm,[-.01,1,3,5,np.inf],labels=False,include_lowest=True).astype(int)
    for k in [0,1,2]: x[f"feature_triage_{k}"]=(x.triage_class==k).astype(int)
    for k in [0,1,2,3]: x[f"feature_mls_bin_{k}"]=(x.mls_bin==k).astype(int)
    return x
BAL=["feature_triage_0","feature_triage_1","feature_triage_2","feature_any_ich","feature_fracture","feature_V_EDH","feature_V_SDH","feature_V_IPH","feature_V_SAH","feature_V_IVH","feature_mls_bin_0","feature_mls_bin_1","feature_mls_bin_2","feature_mls_bin_3"]
def choose(df,n,s0,trials):
    full=df[BAL].mean(); req=["feature_fracture","feature_V_EDH","feature_V_SDH","feature_V_IPH","feature_V_SAH","feature_V_IVH","feature_mls_bin_1","feature_mls_bin_2","feature_mls_bin_3"]; best=None
    for s in range(s0,s0+trials):
        rem,sub=train_test_split(df,test_size=n,random_state=s,shuffle=True,stratify=df.triage_class)
        if any(sub[c].sum()==0 or rem[c].sum()==0 for c in req): continue
        score=float((sub[BAL].mean()-full).abs().mean()+(rem[BAL].mean()-full).abs().mean())
        if best is None or score<best[0]: best=(score,s,rem.copy(),sub.copy())
    if best is None: raise RuntimeError("Split failed.")
    return best
sf=features(targets); _,_,train_dev,test_df=choose(sf,N_TEST,SEED,N_SPLIT_TRIALS); _,_,train_df,dev_df=choose(train_dev.reset_index(drop=True),N_DEV,SEED+N_SPLIT_TRIALS+1,N_SPLIT_TRIALS)
TRAIN_IDS=set(train_df.series_id); DEV_IDS=set(dev_df.series_id)
print("TRAIN",len(TRAIN_IDS),"DEV",len(DEV_IDS))

## 5. Build keypoint slice index

In [ ]:
CACHE=CACHE_DIR/"mls_slice_index.pkl"
def scalar(ds):
    try:
        ori=np.asarray(ds.ImageOrientationPatient,float); pos=np.asarray(ds.ImagePositionPatient,float); return float(np.dot(pos,np.cross(ori[:3],ori[3:6])))
    except Exception: return np.nan
def slice_mls(points,row_spacing,col_spacing):
    p=np.asarray(points,float); q=np.column_stack([p[:,0]*col_spacing,p[:,1]*row_spacing]); a,b,o=q; v=b-a; d=np.linalg.norm(v)
    return 0.0 if d<1e-8 else float(abs(v[0]*(o-a)[1]-v[1]*(o-a)[0])/d)

if CACHE.exists():
    idx_df=pd.read_pickle(CACHE)
else:
    rows=[]
    for series_dir in tqdm(sorted([p for p in TRAINING_DIR.iterdir() if p.is_dir()],key=lambda p:int(p.name)),desc="Indexing"):
        sid=norm_id(series_dir.name)
        if sid not in TRAIN_IDS and sid not in DEV_IDS: continue
        split="TRAIN" if sid in TRAIN_IDS else "DEV"
        for dcm in series_dir.glob("*.dcm"):
            ds=pydicom.dcmread(dcm,stop_before_pixels=True,force=True); uid=str(getattr(ds,"SOPInstanceUID",dcm.stem)); sp=[float(x) for x in getattr(ds,"PixelSpacing",[1,1])]
            ann_path=ANNOTATIONS_DIR/sid/f"{uid}.json"; ann_exists=int(ann_path.exists()); relevant=0; mls=np.nan
            if ann_path.exists():
                with open(ann_path,"r",encoding="utf-8") as f: ann=json.load(f)
                kp=ann.get("keypoints",{}) or {}; names=["AnteriorFalxAttachment","PosteriorFalxAttachment","OutermostPointOfTheFalx"]
                if all(kp.get(n) is not None for n in names):
                    relevant=1; mls=slice_mls([kp[n] for n in names],sp[0],sp[1])
            rows.append({"series_id":sid,"split":split,"dicom_path":str(dcm),"sop_uid":uid,"position":scalar(ds),"instance":float(getattr(ds,"InstanceNumber",np.nan)),"annotation_exists":ann_exists,"relevant":relevant,"slice_mls_mm":mls})
    idx_df=pd.DataFrame(rows); idx_df.to_pickle(CACHE)

groups={}; lookup={}
for sid,g in idx_df.groupby("series_id"):
    g=g.copy()
    if g.position.notna().all(): g=g.sort_values("position")
    elif g.instance.notna().all(): g=g.sort_values("instance")
    else: g=g.sort_values("dicom_path")
    g=g.reset_index(drop=True); groups[sid]=g
    for i,r in g.iterrows(): lookup[(sid,str(r.sop_uid))]=i

train_slices=idx_df[(idx_df.split=="TRAIN")&(idx_df.annotation_exists==1)].copy()
train_slices["t1"]=((train_slices.slice_mls_mm>=1)&(train_slices.relevant==1)).astype(int)
train_slices["t3"]=((train_slices.slice_mls_mm>=3)&(train_slices.relevant==1)).astype(int)
train_slices["t5"]=((train_slices.slice_mls_mm>=5)&(train_slices.relevant==1)).astype(int)
print(train_slices[["relevant","t1","t3","t5"]].sum())

## 6. Dataset and model

In [ ]:
@lru_cache(maxsize=HU_CACHE_SIZE)
def load_hu(path):
    ds=pydicom.dcmread(path,force=True); return np.asarray(apply_modality_lut(ds.pixel_array,ds),dtype=np.float32)
def brain(x):
    lo=BRAIN_CENTER-BRAIN_WIDTH/2; hi=BRAIN_CENTER+BRAIN_WIDTH/2
    t=torch.from_numpy(((np.clip(x,lo,hi)-lo)/(hi-lo)).astype(np.float32))
    return F.interpolate(t[None,None],size=(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False)[0,0]

class MLSDataset(Dataset):
    def __init__(self,df,train=False): self.df=df.reset_index(drop=True); self.train=train
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; g=groups[r.series_id]; ci=lookup[(r.series_id,str(r.sop_uid))]; ids=[min(max(ci+o,0),len(g)-1) for o in CONTEXT_OFFSETS]
        x=torch.stack([brain(load_hu(str(g.iloc[j].dicom_path))) for j in ids])
        if self.train and random.random()<.5: x=torch.flip(x,[-1])
        x=(x-.5)/.25
        y=torch.tensor([float(r.relevant),float(r.t1),float(r.t3),float(r.t5)])
        return x,y,r.series_id

series_counts=train_slices.groupby("series_id").size().to_dict()
rel_counts=train_slices.relevant.value_counts().to_dict()
weights=[(1/series_counts[r.series_id])*(1/rel_counts[int(r.relevant)]) for r in train_slices.itertuples()]
sampler=WeightedRandomSampler(torch.tensor(weights,dtype=torch.double),SAMPLES_PER_EPOCH,replacement=True)
train_loader=DataLoader(MLSDataset(train_slices,True),batch_size=BATCH_SIZE,sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True)

model=efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
model.classifier[1]=nn.Linear(model.classifier[1].in_features,4)
model=model.to(DEVICE)

## 7. Train multi-task ordinal model

In [ ]:
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); scaler=torch.amp.GradScaler("cuda",enabled=DEVICE.type=="cuda")
best=-1; stale=0; best_path=MODELS_DIR/"mls_2p5d_ordinal_efficientnet_b4_best.pth"

def loss_fn(logits,y):
    presence=F.binary_cross_entropy_with_logits(logits[:,0],y[:,0])
    mask=y[:,0]>0.5
    ordinal=F.binary_cross_entropy_with_logits(logits[mask,1:],y[mask,1:]) if mask.any() else logits[:,1:].sum()*0
    return .5*presence+.5*ordinal

@torch.inference_mode()
def infer_dev():
    model.eval(); rows=[]
    for sid in tqdm(sorted(DEV_IDS),desc="DEV series",leave=False):
        g=groups[sid]; out=[]
        for start in range(0,len(g),BATCH_SIZE):
            batch=[]
            for ci in range(start,min(start+BATCH_SIZE,len(g))):
                ids=[min(max(ci+o,0),len(g)-1) for o in CONTEXT_OFFSETS]; x=torch.stack([brain(load_hu(str(g.iloc[j].dicom_path))) for j in ids]); batch.append((x-.5)/.25)
            p=torch.sigmoid(model(torch.stack(batch).to(DEVICE))).cpu().numpy(); out.append(p)
        p=np.concatenate(out); joint=p[:,1:]*p[:,0:1]
        row={"series_id":sid,"true_mls":float(targets.loc[targets.series_id==sid,"MLS_mm"].iloc[0])}
        for j,name in enumerate(["p1","p3","p5"]):
            vals=joint[:,j]; row[name+"_max"]=float(vals.max())
            for k in [3,5,10]:
                kk=min(k,len(vals)); row[f"{name}_top{k}"]=float(np.sort(vals)[-kk:].mean())
        rows.append(row)
    return pd.DataFrame(rows)

for epoch in range(EPOCHS):
    model.train(); losses=[]
    for x,y,_ in tqdm(train_loader,desc=f"epoch {epoch+1}/{EPOCHS}",leave=False):
        x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda",enabled=DEVICE.type=="cuda"):
            loss=loss_fn(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(loss.item())
    dev=infer_dev(); scores=[]
    for target,name in [(1,"p1"),(3,"p3"),(5,"p5")]:
        best_t=0
        for agg in ["max","top3","top5","top10"]:
            col=f"{name}_{agg}"
            for th in np.arange(.05,.96,.05):
                best_t=max(best_t,f1_score((dev.true_mls>=target).astype(int),(dev[col]>=th).astype(int),zero_division=0))
        scores.append(best_t)
    mean_f1=float(np.mean(scores)); print(f"Epoch {epoch+1:02d} loss={np.mean(losses):.4f} mean threshold F1={mean_f1:.4f}")
    if mean_f1>best: best=mean_f1; stale=0; torch.save(model.state_dict(),best_path)
    else: stale+=1
    if stale>=PATIENCE: break

model.load_state_dict(torch.load(best_path,map_location=DEVICE)); model.eval()
print("Best mean threshold F1:",best)

## 8. DEV aggregation and threshold calibration

In [ ]:
dev=infer_dev(); config={}; summary=[]
for target,name in [(1,"p1"),(3,"p3"),(5,"p5")]:
    rows=[]
    truth=(dev.true_mls>=target).astype(int)
    for agg in ["max","top3","top5","top10"]:
        col=f"{name}_{agg}"
        for th in np.arange(.01,1.0,.01):
            pred=(dev[col]>=th).astype(int); tn,fp,fn,tp=confusion_matrix(truth,pred,labels=[0,1]).ravel()
            rows.append({"target_mm":target,"aggregator":agg,"threshold":th,"F1":f1_score(truth,pred,zero_division=0),"FP":fp,"FN":fn})
    table=pd.DataFrame(rows).sort_values(["F1","FN","FP"],ascending=[False,True,True]).reset_index(drop=True)
    bestrow=table.iloc[0]; config[name]={"target_mm":target,"aggregator":bestrow.aggregator,"threshold":float(bestrow.threshold),"F1":float(bestrow.F1)}
    summary.append(bestrow.to_dict())

summary_df=pd.DataFrame(summary); summary_df.to_csv(METRICS_DIR/"ordinal_threshold_summary.csv",index=False); dev.to_csv(METRICS_DIR/"dev_series_scores.csv",index=False)
display(summary_df)

def series_bin(row):
    flags=[]
    for name in ["p1","p3","p5"]:
        c=config[name]; flags.append(float(row[f"{name}_{c['aggregator']}"])>=c["threshold"])
    if flags[2]: return 5.5
    if flags[1]: return 3.5
    if flags[0]: return 1.5
    return 0.0

dev["pred_MLS_mm"]=dev.apply(series_bin,axis=1)
dev["true_bin"]=pd.cut(dev.true_mls,[-.01,1,3,5,np.inf],labels=[0,1,2,3],include_lowest=True).astype(int)
dev["pred_bin"]=pd.cut(dev.pred_MLS_mm,[-.01,1,3,5,np.inf],labels=[0,1,2,3],include_lowest=True).astype(int)
print("DEV bin accuracy:",float((dev.true_bin==dev.pred_bin).mean()))

## 9. Save

In [ ]:
torch.save(model.state_dict(),MODELS_DIR/"mls_2p5d_ordinal_efficientnet_b4_final.pth")
with open(MODELS_DIR/"mls_2p5d_ordinal_config.json","w") as f: json.dump({"architecture":"efficientnet_b4","input":"previous_center_next_brain","image_size":IMAGE_SIZE,"brain_window":[BRAIN_CENTER,BRAIN_WIDTH],"threshold_config":config},f,indent=2)
dev[["series_id","true_mls","pred_MLS_mm"]].to_csv(METRICS_DIR/"dev_mls_predictions.csv",index=False)
pd.DataFrame([{"mean_threshold_F1":float(np.mean([config[k]["F1"] for k in config])),"DEV_bin_accuracy":float((dev.true_bin==dev.pred_bin).mean())}]).to_csv(OUTPUT_ROOT/"00_DIRECT_ANSWERS.csv",index=False)